# Supervised Model Development

This notebook implements the supervised-learning workflow for predicting whether a banking complaint will receive an untimely response. The primary source file is `data/processed/consumer_banking_complaints.parquet`.

In [22]:
# pip install -r ../requirements.txt

## Notebook Overview

In this notebook, we will

1. load and quality-check the complaint data
2. engineer leakage-aware structured features plus simple narrative indicators
3. create a stratified train/test split for an imbalanced target
4. tune and compare four model families: Logistic Regression, KNN, Random Forest, and Support Vector Machine
5. select a final development model and save artifacts for downstream evaluation

The target is defined as `target_untimely = 1` when `Timely response? == False`. This reframes the task around the minority class that is typically most operationally important.

## Table of Contents

1. [Data Loading and Exploration](#Data-Loading-and-Exploration)
2. [Feature Engineering](#Feature-Engineering)
3. [Train/Test Setup](#Train/Test-Setup)
4. [Model #1 Logistic Regression](#Model-#1-Logistic-Regression)
5. [Model #2 KNN](#Model-#2-KNN)
6. [Model #3 Random Forest](#Model-#3-Random-Forest)
7. [Model #4 Support Vector Machine](#Model-#4-Support-Vector-Machine)
8. [Final Model Selection](#Final-Model-Selection)

### Data Loading and Exploration

We start by validating that the banking-complaints parquet is internally consistent enough for supervised learning. Before training any model, we check for duplicate complaint identifiers, target completeness, date parsing issues, negative date gaps, and the amount of missingness in high-value fields such as the narrative.

In [23]:
import os
from pathlib import Path
import warnings

import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "consumer_banking_complaints.parquet"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "processed"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)

MODEL_SCOPE = "all_banking"
MAX_MODEL_ROWS = 60_000
KNN_MAX_TRAIN_ROWS = 20_000
RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = min(4, max(1, (os.cpu_count() or 2) - 1))

assert DATA_PATH.exists(), f"Expected source parquet at {DATA_PATH}"

raw_df = pd.read_parquet(DATA_PATH)

print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns from {DATA_PATH.name}")
raw_df.head()

Loaded 1,133,355 rows and 18 columns from consumer_banking_complaints.parquet


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,True,N/A,3477549
1,2019-12-20,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,FL,33064,NaN,N/A,Referral,2019-12-23,Closed with explanation,True,N/A,3475858
2,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136
3,2020-06-05,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,10466,NaN,Consent not provided,Web,2020-06-05,Closed with explanation,True,N/A,3684669
4,2024-01-16,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Add-on products and services,NaN,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,TX,76179,NaN,Consent not provided,Web,2024-01-16,Closed with monetary relief,True,N/A,8161600


In [24]:
quality_df = raw_df.copy()
quality_df["Date received"] = pd.to_datetime(quality_df["Date received"], errors="coerce")
quality_df["Date sent to company"] = pd.to_datetime(quality_df["Date sent to company"], errors="coerce")
quality_df["company_lag_days"] = (
    quality_df["Date sent to company"] - quality_df["Date received"]
).dt.days

quality_summary = pd.DataFrame(
    {
        "dtype": raw_df.dtypes.astype(str),
        "missing_count": raw_df.isna().sum(),
        "missing_pct": (raw_df.isna().mean() * 100).round(2),
        "n_unique": raw_df.nunique(dropna=True),
    }
).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

data_quality_checks = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "duplicate_rows": int(raw_df.duplicated().sum()),
        "duplicate_complaint_ids": int(raw_df["Complaint ID"].duplicated().sum()),
        "null_target_count": int(raw_df["Timely response?"].isna().sum()),
        "unparseable_date_received": int(quality_df["Date received"].isna().sum()),
        "unparseable_date_sent": int(quality_df["Date sent to company"].isna().sum()),
        "negative_lag_rows": int((quality_df["company_lag_days"] < 0).sum()),
        "narrative_missing_pct": round(raw_df["Consumer complaint narrative"].isna().mean() * 100, 2),
        "target_untimely_pct": round((~raw_df["Timely response?"].astype(bool)).mean() * 100, 3),
    },
    name="value",
)

display(data_quality_checks.to_frame())
display(quality_summary.head(12))
display(quality_df["company_lag_days"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame().T)

,value
rows,1133355.000
columns,18.000
duplicate_rows,0.000
duplicate_complaint_ids,0.000
null_target_count,0.000
unparseable_date_received,0.000
unparseable_date_sent,0.000
negative_lag_rows,0.000
narrative_missing_pct,52.250
target_untimely_pct,1.153


,dtype,missing_count,missing_pct,n_unique
Tags,str,931835,82.22,3
Consumer complaint narrative,str,592147,52.25,523051
Company public response,str,570806,50.36,11
Sub-issue,str,264229,23.31,130
Consumer consent provided?,str,49622,4.38,5
Sub-product,str,28299,2.50,28
State,str,19066,1.68,63
ZIP code,str,13872,1.22,25542
Complaint ID,int64,0,0.00,1133355
Date received,object,0,0.00,3800


,count,mean,std,min,1%,5%,50%,95%,99%,max
company_lag_days,1133355.0,1.685287,7.595387,0.0,0.0,0.0,0.0,8.0,36.0,543.0


In [25]:
product_summary = (
    raw_df.groupby("Product", dropna=False)
    .agg(
        complaints=("Complaint ID", "count"),
        untimely_rate=("Timely response?", lambda s: (~s.astype(bool)).mean()),
        narrative_available=("Consumer complaint narrative", lambda s: s.notna().mean()),
    )
    .sort_values("complaints", ascending=False)
)
product_summary[["untimely_rate", "narrative_available"]] = (
    product_summary[["untimely_rate", "narrative_available"]] * 100
).round(2)

display(product_summary)
display(raw_df["Timely response?"].value_counts(dropna=False).rename("count").to_frame())

high_null_columns = quality_summary.loc[quality_summary["missing_pct"] >= 20, ["missing_pct", "n_unique"]]
display(high_null_columns)

,complaints,untimely_rate,narrative_available
Product,,,
Checking or savings account,368334,0.97,48.48
Mortgage,275705,1.50,47.49
Credit card,254151,0.52,44.36
Credit card or prepaid card,206364,1.18,52.66
Bank account or service,28801,5.54,35.85


,count
Timely response?,
True,1120289
False,13066


,missing_pct,n_unique
Tags,82.22,3
Consumer complaint narrative,52.25,523051
Company public response,50.36,11
Sub-issue,23.31,130


The initial quality check surfaces a few important findings that shape the rest of the notebook:

- The processed banking dataset is large enough for supervised learning at over 1.13 million complaints across 18 columns.
- `Complaint ID` appears unique and the date fields parse cleanly, which reduces the need for aggressive record repair.
- The target is highly imbalanced, with untimely responses making up only about 1.15% of complaints, so accuracy alone would be misleading.
- Narrative coverage is meaningful but incomplete at roughly 47.75%, which motivates using both text-derived features and non-text structured features.
- Missingness is concentrated in fields like `Tags`, `Company public response`, `Consumer complaint narrative`, and `Sub-issue`, so our preprocessing needs robust imputers and careful leakage handling.

The product-level summary also confirms that this is a heterogeneous all-banking problem rather than a single-product classification task. That heterogeneity is one reason we preserve `Product`, `Issue`, channel, geography, and company context as predictive inputs.

### Feature Engineering

We intentionally exclude fields that are likely to leak post-submission information into the prediction task. In particular, `Date sent to company`, `Company response to consumer`, `Company public response`, and `Consumer disputed?` are better suited for diagnostics than for pre-response prediction.

Our feature design tries to balance predictive power with realism. If a feature is only known after the complaint has already been processed by the company, we exclude it from model training even if it would make prediction easier.

Narrative information in this notebook is intentionally kept simple:

- whether a narrative is present
- basic narrative length measures such as character and word count

More advanced narrative modeling is intentionally left out of this notebook so it does not overlap with the separate unsupervised clustering and feature-generation workstream.

In [26]:
def make_one_hot_encoder(dense=False):
    kwargs = {"handle_unknown": "ignore"}
    try:
        return OneHotEncoder(sparse_output=not dense, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=not dense, **kwargs)


def score_estimator(estimator, X_test, y_test):
    if hasattr(estimator, "predict_proba"):
        y_score = estimator.predict_proba(X_test)[:, 1]
    else:
        y_score = estimator.decision_function(X_test)
    y_pred = estimator.predict(X_test)
    return {
        "holdout_accuracy": accuracy_score(y_test, y_pred),
        "holdout_average_precision": average_precision_score(y_test, y_score),
        "holdout_roc_auc": roc_auc_score(y_test, y_score),
        "holdout_recall": recall_score(y_test, y_pred, zero_division=0),
        "holdout_precision": precision_score(y_test, y_pred, zero_division=0),
        "holdout_f1": f1_score(y_test, y_pred, zero_division=0),
        "holdout_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "y_score": y_score,
        "y_pred": y_pred,
    }


def build_results_row(model_name, search, X_test, y_test):
    best_idx = search.best_index_
    metric_names = [
        "accuracy",
        "average_precision",
        "roc_auc",
        "recall",
        "precision",
        "f1",
        "balanced_accuracy",
    ]
    row = {
        "model_family": model_name,
        "best_params": search.best_params_,
    }
    for metric in metric_names:
        row[f"cv_mean_{metric}"] = search.cv_results_[f"mean_test_{metric}"][best_idx]
        row[f"cv_std_{metric}"] = search.cv_results_[f"std_test_{metric}"][best_idx]
    holdout_scores = score_estimator(search.best_estimator_, X_test, y_test)
    row.update({k: v for k, v in holdout_scores.items() if not k.startswith("y_")})
    return row, holdout_scores


leakage_columns = [
    "Timely response?",
    "Complaint ID",
    "Date sent to company",
    "Company response to consumer",
    "Company public response",
    "Consumer disputed?",
    "Consumer consent provided?",
]

model_df = raw_df.copy()
model_df["Date received"] = pd.to_datetime(model_df["Date received"], errors="coerce")
model_df["target_untimely"] = (~model_df["Timely response?"].astype(bool)).astype(int)

text_series = model_df["Consumer complaint narrative"].fillna("")
model_df["narrative_present"] = text_series.str.len().gt(0).astype(int)
model_df["narrative_char_count"] = text_series.str.len()
model_df["narrative_word_count"] = text_series.str.split().str.len().fillna(0)

model_df["received_year"] = model_df["Date received"].dt.year
model_df["received_month"] = model_df["Date received"].dt.month
model_df["received_quarter"] = model_df["Date received"].dt.quarter
model_df["received_dayofweek"] = model_df["Date received"].dt.dayofweek
model_df["received_day"] = model_df["Date received"].dt.day
model_df["received_days_since_start"] = (
    model_df["Date received"] - model_df["Date received"].min()
).dt.days
model_df["zip3"] = (
    model_df["ZIP code"].fillna("").astype(str).str.extract(r"(\d{3})", expand=False).fillna("missing")
)

top_category_limits = {
    "Company": 75,
    "Sub-product": 30,
    "Issue": 40,
    "Sub-issue": 60,
    "State": 30,
    "Tags": 10,
    "zip3": 100,
}
for col, top_n in top_category_limits.items():
    values = model_df[col].fillna("missing").astype(str)
    keep = set(values.value_counts().head(top_n).index)
    model_df[col] = values.where(values.isin(keep), other="__OTHER__")
model_df["Submitted via"] = model_df["Submitted via"].fillna("missing").astype(str)
model_df["Product"] = model_df["Product"].fillna("missing").astype(str)

model_df = model_df.dropna(subset=["Date received", "target_untimely"]).copy()

if len(model_df) > MAX_MODEL_ROWS:
    _, model_df = train_test_split(
        model_df,
        test_size=MAX_MODEL_ROWS,
        stratify=model_df["target_untimely"],
        random_state=RANDOM_STATE,
    )

print(f"Modeling rows after optional down-sampling: {len(model_df):,}")
print(f"Untimely class rate: {model_df['target_untimely'].mean() * 100:.3f}%")
display(pd.Series(leakage_columns, name="excluded_from_modeling"))

Modeling rows after optional down-sampling: 60,000
Untimely class rate: 1.153%


0                Timely response?
1                    Complaint ID
2            Date sent to company
3    Company response to consumer
4         Company public response
5              Consumer disputed?
6      Consumer consent provided?
Name: excluded_from_modeling, dtype: str

A few design choices are worth emphasizing here:

- We keep only lightweight narrative indicators in this notebook, specifically whether a narrative exists and how long it is.
- We convert the prediction target to `target_untimely`, making the rare failure case the positive class. That makes recall, precision, and average precision easier to interpret.
- We include simple calendar features from `Date received` because operational timing effects can matter, but we exclude `Date sent to company` because it is too close to the response workflow and could leak downstream process information.
- We down-sample only for tractability, not because the source data is low quality. The sample remains stratified so the rare-class rate stays representative.

### Train/Test Setup

We use a stratified split and 5-fold stratified cross-validation because the untimely-response class is rare. We report accuracy, precision, recall, F1, ROC AUC, balanced accuracy, and average precision for each model, but the model-selection and cross-fold comparison criterion is F1. We also use SMOTE inside the dense-feature KNN pipeline, while the linear Logistic Regression and SVM models use class weighting instead.

In [27]:
target_col = "target_untimely"

structured_numeric = [
    "received_year",
    "received_month",
    "received_quarter",
    "received_dayofweek",
    "received_day",
    "received_days_since_start",
    "narrative_present",
    "narrative_char_count",
    "narrative_word_count",
]

logistic_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "Tags",
    "zip3",
]

knn_categorical = ["Product", "Issue", "State", "Submitted via"]
rf_categorical = ["Product", "Sub-product", "Issue", "Sub-issue", "State", "Submitted via", "Tags", "zip3"]

logistic_features = logistic_categorical + structured_numeric
knn_features = knn_categorical + structured_numeric
rf_features = rf_categorical + structured_numeric

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)

scoring = {
    "accuracy": "accuracy",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train untimely rate: {train_df[target_col].mean() * 100:.3f}%")
print(f"Test untimely rate: {test_df[target_col].mean() * 100:.3f}%")
print(f"Grid-search workers: {N_JOBS}")

Train rows: 48,000
Test rows: 12,000
Train untimely rate: 1.154%
Test untimely rate: 1.150%
Grid-search workers: 4


### Model #1 Logistic Regression

This linear baseline is a strong fit for one-hot encoded categorical features plus the simple narrative indicators. We tune regularization strength while keeping `class_weight='balanced'` to counter the strong class imbalance.

Design expectation: Logistic Regression should remain a strong and interpretable baseline because it handles high-cardinality categorical features well and produces stable decision boundaries.

In [28]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            logistic_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                solver="saga",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_grid = {
    "model__C": [0.5, 1.0, 2.0],
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_log = train_df[logistic_features]
y_train_log = train_df[target_col]
X_test_log = test_df[logistic_features]
y_test_log = test_df[target_col]

logistic_search.fit(X_train_log, y_train_log)
logistic_row, logistic_scores = build_results_row("Logistic Regression", logistic_search, X_test_log, y_test_log)

logistic_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", target_col]].copy()
logistic_holdout["model_family"] = "Logistic Regression"
logistic_holdout["score"] = logistic_scores["y_score"]
logistic_holdout["prediction"] = logistic_scores["y_pred"]

display(pd.DataFrame([logistic_row]).T)

Fitting 5 folds for each of 3 candidates, totalling 15 fits


,0
model_family,Logistic Regression
best_params,{'model__C': 0.5}
cv_mean_accuracy,0.699979
cv_std_accuracy,0.005383
cv_mean_average_precision,0.027143
cv_std_average_precision,0.002976
cv_mean_roc_auc,0.642191
cv_std_roc_auc,0.021006
cv_mean_recall,0.500066
cv_std_recall,0.031542


### Model #2 KNN

KNN gives us a very different, instance-based learning family. Because distance-based models scale poorly in very high dimensions, this version uses compact structured features plus simple narrative indicators rather than raw text. That dense feature space also makes it a good candidate for SMOTE within cross-validation.

Design expectation: KNN is less likely to dominate on this task, but it provides a useful contrast because it relies on local neighborhood structure rather than a global linear or tree-based decision rule.

In [29]:
if len(train_df) > KNN_MAX_TRAIN_ROWS:
    _, knn_train_df = train_test_split(
        train_df,
        test_size=KNN_MAX_TRAIN_ROWS,
        stratify=train_df[target_col],
        random_state=RANDOM_STATE,
    )
else:
    knn_train_df = train_df.copy()

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_one_hot_encoder(dense=True)),
                ]
            ),
            knn_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

knn_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", knn_preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.20, k_neighbors=3)),
        ("model", KNeighborsClassifier()),
    ]
)

knn_grid = {
    "smote__sampling_strategy": [0.10, 0.20],
    "smote__k_neighbors": [3, 5],
    "model__n_neighbors": [15, 35, 75],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_knn = knn_train_df[knn_features]
y_train_knn = knn_train_df[target_col]
X_test_knn = test_df[knn_features]
y_test_knn = test_df[target_col]

knn_search.fit(X_train_knn, y_train_knn)
knn_row, knn_scores = build_results_row("KNN", knn_search, X_test_knn, y_test_knn)

knn_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", target_col]].copy()
knn_holdout["model_family"] = "KNN"
knn_holdout["score"] = knn_scores["y_score"]
knn_holdout["prediction"] = knn_scores["y_pred"]

display(pd.DataFrame([knn_row]).T)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


,0
model_family,KNN
best_params,"{'model__n_neighbors': 75, 'model__p': 1, 'mod..."
cv_mean_accuracy,0.9754
cv_std_accuracy,0.001602
cv_mean_average_precision,0.039324
cv_std_average_precision,0.009038
cv_mean_roc_auc,0.585326
cv_std_roc_auc,0.022927
cv_mean_recall,0.095282
cv_std_recall,0.022327


### Model #3 Random Forest

Random Forest gives us a tree-based, nonlinear baseline. It can capture interactions among the engineered structured and narrative-summary features without assuming linear relationships. For this wide one-hot encoded feature space we keep `class_weight='balanced_subsample'` rather than vanilla SMOTE, which would be much more memory-intensive after expansion.

Design expectation: Random Forest can capture useful nonlinear interactions among issue type, submission channel, timing, and simple narrative indicators, but it may trade off some interpretability relative to the linear baselines.

In [30]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            rf_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            structured_numeric,
        ),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS,
            ),
        ),
    ]
)

rf_grid = {
    "model__max_depth": [None, 12, 20],
    "model__min_samples_leaf": [1, 5, 20],
    "model__max_features": ["sqrt", 0.5],
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_rf = train_df[rf_features]
y_train_rf = train_df[target_col]
X_test_rf = test_df[rf_features]
y_test_rf = test_df[target_col]

rf_search.fit(X_train_rf, y_train_rf)
rf_row, rf_scores = build_results_row("Random Forest", rf_search, X_test_rf, y_test_rf)

rf_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", target_col]].copy()
rf_holdout["model_family"] = "Random Forest"
rf_holdout["score"] = rf_scores["y_score"]
rf_holdout["prediction"] = rf_scores["y_pred"]

display(pd.DataFrame([rf_row]).T)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,0
model_family,Random Forest
best_params,"{'model__max_depth': 20, 'model__max_features'..."
cv_mean_accuracy,0.975833
cv_std_accuracy,0.00182
cv_mean_average_precision,0.045857
cv_std_average_precision,0.012208
cv_mean_roc_auc,0.694506
cv_std_roc_auc,0.038771
cv_mean_recall,0.122654
cv_std_recall,0.027441


### Model #4 Support Vector Machine

Support Vector Machines add a margin-based model family that is distinct from both probabilistic linear models and tree-based methods. We use `LinearSVC` rather than a kernel SVM because the feature space is still fairly wide after one-hot encoding the categorical variables.

Design expectation: if the decision boundary is mostly linear but benefits from maximum-margin separation rather than calibrated probabilities, the SVM may perform competitively on F1 and recall.

In [31]:
svm_categorical = [
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "Tags",
    "zip3",
]
svm_features = svm_categorical + structured_numeric

svm_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_one_hot_encoder(dense=False)),
                ]
            ),
            svm_categorical,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            structured_numeric,
        ),
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", svm_preprocessor),
        ("model", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, dual="auto", max_iter=5000)),
    ]
)

svm_grid = {
    "model__C": [0.25, 0.5, 1.0],
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=N_JOBS,
    verbose=1,
)

X_train_svm = train_df[svm_features]
y_train_svm = train_df[target_col]
X_test_svm = test_df[svm_features]
y_test_svm = test_df[target_col]

svm_search.fit(X_train_svm, y_train_svm)
svm_row, svm_scores = build_results_row("Support Vector Machine", svm_search, X_test_svm, y_test_svm)

svm_holdout = test_df[["Complaint ID", "Product", "Issue", "Company", target_col]].copy()
svm_holdout["model_family"] = "Support Vector Machine"
svm_holdout["score"] = svm_scores["y_score"]
svm_holdout["prediction"] = svm_scores["y_pred"]

display(pd.DataFrame([svm_row]).T)

Fitting 5 folds for each of 3 candidates, totalling 15 fits


,0
model_family,Support Vector Machine
best_params,{'model__C': 0.25}
cv_mean_accuracy,0.696354
cv_std_accuracy,0.00483
cv_mean_average_precision,0.025791
cv_std_average_precision,0.002227
cv_mean_roc_auc,0.638088
cv_std_roc_auc,0.021541
cv_mean_recall,0.501851
cv_std_recall,0.03272


### Final Model Selection

We compare the best tuned model from each family using mean and standard deviation across cross-validation folds, then confirm the selected model on the holdout split.

Interpretation guidance: because the positive class is rare, the notebook now uses cross-validated F1 as the model-selection criterion. We still report accuracy, precision, recall, ROC AUC, balanced accuracy, and average precision so you can discuss the tradeoffs in the evaluation notebook.

In [32]:
results_df = pd.DataFrame([logistic_row, knn_row, rf_row, svm_row]).sort_values(
    by="cv_mean_f1", ascending=False
).reset_index(drop=True)

comparison_columns = [
    "model_family",
    "cv_mean_accuracy",
    "cv_std_accuracy",
    "cv_mean_average_precision",
    "cv_std_average_precision",
    "cv_mean_roc_auc",
    "cv_std_roc_auc",
    "cv_mean_recall",
    "cv_std_recall",
    "cv_mean_precision",
    "cv_std_precision",
    "cv_mean_f1",
    "cv_std_f1",
    "holdout_accuracy",
    "holdout_average_precision",
    "holdout_roc_auc",
    "holdout_recall",
    "holdout_precision",
    "holdout_f1",
]
display(results_df[comparison_columns])

searches = {
    "Logistic Regression": logistic_search,
    "KNN": knn_search,
    "Random Forest": rf_search,
    "Support Vector Machine": svm_search,
}
holdout_frames = {
    "Logistic Regression": logistic_holdout,
    "KNN": knn_holdout,
    "Random Forest": rf_holdout,
    "Support Vector Machine": svm_holdout,
}

best_model_name = results_df.loc[0, "model_family"]
best_search = searches[best_model_name]
best_holdout = holdout_frames[best_model_name].sort_values("score", ascending=False).reset_index(drop=True)

print(f"Selected development model: {best_model_name}")
print(f"Best params: {best_search.best_params_}")

comparison_path = ARTIFACT_DIR / "supervised_model_comparison.csv"
visuals_comparison_path = VISUALS_DIR / "03_model_comparison.csv"
holdout_path = ARTIFACT_DIR / "best_model_holdout_predictions.parquet"
model_path = ARTIFACT_DIR / "best_timely_response_model.joblib"

results_df.to_csv(comparison_path, index=False)
results_df[comparison_columns].round(4).to_csv(visuals_comparison_path, index=False)
best_holdout.to_parquet(holdout_path, index=False)
joblib.dump(best_search.best_estimator_, model_path)

print(f"Saved comparison table to {comparison_path}")
print(f"Saved GitHub-friendly comparison table to {visuals_comparison_path}")
print(f"Saved holdout predictions to {holdout_path}")
print(f"Saved trained model to {model_path}")

best_holdout.head(10)

,model_family,cv_mean_accuracy,cv_std_accuracy,cv_mean_average_precision,cv_std_average_precision,cv_mean_roc_auc,cv_std_roc_auc,cv_mean_recall,cv_std_recall,cv_mean_precision,cv_std_precision,cv_mean_f1,cv_std_f1,holdout_accuracy,holdout_average_precision,holdout_roc_auc,holdout_recall,holdout_precision,holdout_f1
0,Random Forest,0.975833,0.001820,0.045857,0.012208,0.694506,0.038771,0.122654,0.027441,0.092650,0.023940,0.105252,0.024979,0.974083,0.031176,0.673845,0.101449,0.069652,0.082596
1,KNN,0.975400,0.001602,0.039324,0.009038,0.585326,0.022927,0.095282,0.022327,0.071546,0.010059,0.081371,0.013879,0.974583,0.017320,0.572069,0.057971,0.043716,0.049844
2,Logistic Regression,0.699979,0.005383,0.027143,0.002976,0.642191,0.021006,0.500066,0.031542,0.019220,0.000830,0.037017,0.001625,0.701083,0.019772,0.596515,0.427536,0.016541,0.031849
3,Support Vector Machine,0.696354,0.004830,0.025791,0.002227,0.638088,0.021541,0.501851,0.032720,0.019056,0.000895,0.036718,0.001749,0.699917,0.018699,0.593203,0.427536,0.016476,0.031729


Selected development model: Random Forest
Best params: {'model__max_depth': 20, 'model__max_features': 0.5, 'model__min_samples_leaf': 20}
Saved comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\supervised_model_comparison.csv
Saved GitHub-friendly comparison table to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\visuals\03_model_comparison.csv
Saved holdout predictions to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\best_model_holdout_predictions.parquet
Saved trained model to C:\Users\ntamm\OneDrive\Documents\1_MADS\1_Classes\696 - Milestone 2\Project\cfpb_mortgage_complaints\data\processed\best_timely_response_model.joblib


,Complaint ID,Product,Issue,Company,target_untimely,model_family,score,prediction
0,2148471,Bank account or service,"Account opening, closing, or management","CITIBANK, N.A.",0,Random Forest,0.950064,1
1,2148478,Bank account or service,"Account opening, closing, or management",JPMORGAN CHASE & CO.,0,Random Forest,0.924641,1
2,2151772,Bank account or service,Deposits and withdrawals,WELLS FARGO & COMPANY,0,Random Forest,0.902695,1
3,2158524,Bank account or service,"Account opening, closing, or management",WELLS FARGO & COMPANY,0,Random Forest,0.890678,1
4,2202742,Bank account or service,"Making/receiving payments, sending money",U.S. BANCORP,0,Random Forest,0.859466,1
5,2247966,Bank account or service,Deposits and withdrawals,JPMORGAN CHASE & CO.,0,Random Forest,0.852793,1
6,2145762,Bank account or service,Deposits and withdrawals,CAPITAL ONE FINANCIAL CORPORATION,0,Random Forest,0.850419,1
7,2162245,Bank account or service,"Account opening, closing, or management",TD BANK US HOLDING COMPANY,0,Random Forest,0.838930,1
8,2282468,Bank account or service,"Account opening, closing, or management",DISCOVER BANK,0,Random Forest,0.831080,1
9,3609596,Credit card or prepaid card,Problem getting a card or closing an account,U.S. BANCORP,0,Random Forest,0.825618,1


### Narrative Ideas For Future Work

This supervised notebook intentionally avoids more complex narrative engineering, but these would be strong candidates for a separate narrative-focused workstream:

- topic clusters or complaint themes from unsupervised text clustering
- sentence embeddings from transformer-based encoders
- sentiment, emotion, or urgency indicators
- keyword groups tied to servicing delays, escrow, fraud, fees, or payment-processing issues
- readability or complexity measures
- named entities such as company names, products, locations, or dollar amounts
- temporal drift in narrative themes over time
- cluster membership or distance-to-centroid features merged back into supervised models